In [ ]:
from huggingface_hub import login
login(token="https://github.com/Bao-Bao-03/VGU_WS26_ClinicalProject_BHTBH")  # Paste self's access token in here

Check if access token is actually active:

In [ ]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6aad11231339b5a189922a50', 'name': 'choisongjun', 'fullname': 'HT', 'isPro': False, 'avatarUrl': '/avatars/481c7e7c8d0b9c678adf6db8ab7a351d.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'MedGemmaToken-1', 'role': 'fineGrained', 'createdAt': '2026-09-19T06:17:45.673Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6aad11231339b5a189922a50', 'type': 'user', 'name': 'choisongjun'}, 'permissions': ['repo.content.read']}]}}}}


Check how much VRAM is available:

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.total,memory.used,memory.free", "--format=csv"], capture_output=True, text=True).stdout)

memory.total [MiB], memory.used [MiB], memory.free [MiB]
6144 MiB, 63 MiB, 5744 MiB



Import MedGemma from HuggingFace's model hub.

**Quantization** to shrink memory footprint in order to fit GPU's VRAM:


Note: Run `pip install torchao accelerate` before running this cell

In [ ]:
from transformers import pipeline, TorchAoConfig
from torchao.quantization import Int4WeightOnlyConfig

model_id = "google/medgemma-4b-it"

quantization_config = TorchAoConfig(Int4WeightOnlyConfig(group_size=32))

pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0919 14:34:13.852000 9316 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0919 14:34:13.880000 9316 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights:  34%|███▍      | 303/883 [00:04<00:06, 91.71it/s][W919 14:34:19.74466901

If running the above cell leads to memory errors, run this instead (`pip install bitsandbytes`)

In [ ]:
from transformers import pipeline, BitsAndBytesConfig
import torch

model_id = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    model_kwargs={
        "quantization_config": quantization_config,
        "low_cpu_mem_usage": True,
    },
    device_map="auto",
)

/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0919 14:39:42.236000 10558 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0919 14:39:42.264000 10558 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 883/883 [00:06<00:00, 143.56it/s]


Test model by feeding an image to it

In [ ]:
from PIL import Image

image_path = "/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/Demos/medical_datasets/_tmp_dengue/images/PMC1/PMC10/PMC10225563_fimmu-14-1129246-g001_A_1_2.webp"
image = Image.open(image_path)   # raises a clear error immediately if the path is wrong

messages = [
    {"role": "system", "content": [{"type": "text", "text": "You are a helpful medical assistant."}]},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Symptoms: fever, rash for 3 days. What's the likely diagnosis?"},
    ]},
]

output = pipe(text=messages, max_new_tokens=200)
print(output[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the image and the reported symptoms of fever and rash for 3 days, the most likely diagnosis is **impetigo**.

Here's why:

*   **Impetigo:** This is a common bacterial skin infection, often caused by *Staphylococcus aureus* or *Streptococcus pyogenes*. It typically presents with honey-colored, crusted lesions that can be itchy and sometimes have a central black dot. The rash can be widespread and can occur on the face, hands, and feet. The fever is also consistent with an infection.

Other possibilities, though less likely, could include:

*   **Cellulitis:** This is a deeper skin infection that can cause redness, swelling, warmth, and pain. It can be associated with fever.
*   **Erysipelas:** A more superficial form of cellulitis, often caused by *Streptococcus pyogenes*, that presents with a well-defined, raised, and red rash.

